# ShopFlow — Phase 5: Streams, Tasks & Sharing
### CDC with streams · nightly automation with tasks · live data sharing

---

### What this notebook does

Three Snowflake features that replace entire AWS service stacks with pure SQL:

| Feature | What it does | AWS equivalent |
|---------|-------------|----------------|
| `STREAM` | Tracks every change to a table since last consumed — Change Data Capture | DynamoDB Streams + Debezium on RDS |
| `TASK` | Runs SQL on a CRON schedule or when triggered | EventBridge + Lambda |
| `SHARE` | Exposes tables to another Snowflake account — no data copy | S3 + pre-signed URLs + Glue ETL |

### The automation pipeline you will build

```text
FACT_ORDERS  →  ORDERS_STREAM  →  NIGHTLY_SELLER_SUMMARY task
     (table)        (CDC)               (CRON 01:00 Asia/Kolkata)
                                              ↓
                                  DAILY_SELLER_SUMMARY table
                                              ↓
                                  BRAND_PARTNER_SHARE
                                    (live, no data copy)
```

A seller partner logs into a reader account and queries `DAILY_SELLER_SUMMARY` in real time — no SFTP, no email attachments, no nightly CSV exports.

### What makes this different from AWS

In AWS, this pipeline requires: RDS + Debezium (CDC) → Kinesis → Lambda → S3 → Glue → Redshift → API Gateway → partner access. Every component has its own IAM, monitoring, and deployment pipeline.

In Snowflake: four SQL statements.

---

### Run cells one at a time — top to bottom

Each cell = one statement = one visible result.

---

### Key Snowflake concepts in this notebook

| Topic | Cells |
|-------|-------|
| `CREATE STREAM` — standard vs append-only | Cell 2–4 |
| Stream offset — how it advances | Cell 4–5 |
| `METADATA$ACTION`, `METADATA$ISUPDATE` | Cell 4 |
| `SYSTEM$STREAM_HAS_DATA()` | Cell 5 |
| `CREATE TASK` — warehouse vs serverless | Cell 6–9 |
| CRON syntax in Snowflake | Cell 7 |
| `WHEN` clause — conditional task execution | Cell 7 |
| Task DAGs — chaining tasks | Cell 8 |
| `TASK_HISTORY` in `INFORMATION_SCHEMA` | Cell 9 |
| `CREATE SHARE` — provider side | Cell 10–12 |
| `SHOW SHARES` / `SHOW GRANTS TO SHARE` | Cell 12 |
| Snowflake Marketplace context | Cell 10 |


---
## Cell 1 — Set context

Phase 5 objects (stream, task, summary table, share) all live in `SHOPFLOW_ANALYTICS` — the layer closest to consumers.


In [ ]:
USE ROLE      SYSADMIN;
--USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE  SHOPFLOW_DB;
USE SCHEMA    SHOPFLOW_ANALYTICS;


In [ ]:
SELECT
    CURRENT_ROLE()      AS role,
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_DATABASE()  AS database,
    CURRENT_SCHEMA()    AS schema;


---
## Cells 2–5 — Streams (Change Data Capture)

### What is a stream?

A **stream** is a change tracking object attached to a table. Every time a row is inserted, updated, or deleted in the source table, the stream records what changed. When you query the stream, you see only the rows that changed since the last time the stream was consumed.

Think of it as a bookmark in a log. The bookmark only advances when a DML statement reads from the stream — not when you `SELECT` from it.

```text
FACT_ORDERS table
     │
     │  new rows inserted over time
     ▼
ORDERS_STREAM
     │
     │  only rows added/changed since last consume
     ▼
query or task reads stream → bookmark advances → stream is empty again
```

### Standard stream vs append-only stream

| | Standard stream | Append-only stream |
|--|----------------|-------------------|
| Tracks | INSERT + UPDATE + DELETE | INSERT only |
| `METADATA$ACTION` values | `INSERT` and `DELETE` | `INSERT` only |
| Performance | Slightly more overhead | More efficient |
| Use case | Tables with updates and deletes | Append-only fact tables |

`FACT_ORDERS` is append-only in our project — orders are inserted, never updated or deleted. So we use `APPEND_ONLY = TRUE`.

### The three metadata columns every stream adds

| Column | Type | Meaning |
|--------|------|---------|
| `METADATA$ACTION` | VARCHAR | `INSERT` or `DELETE` |
| `METADATA$ISUPDATE` | BOOLEAN | `TRUE` if this row is part of an UPDATE operation |
| `METADATA$ROW_ID` | VARCHAR | Unique ID for the changed row |

> **🎯 SnowPro Tip: Stream Offset**
>
> The stream offset only advances when a **DML statement** (INSERT, UPDATE, DELETE, MERGE) reads from the stream. A plain `SELECT` from the stream does NOT advance it — you can preview stream contents without consuming them. Only a task or an explicit INSERT...SELECT FROM stream will advance the offset.
>
> `SYSTEM$STREAM_HAS_DATA('stream_name')` — returns TRUE if the stream has unconsumed records. Use this as the `WHEN` condition in a task so the task only runs when there is actually new data.


---
## Cell 2 — Create DAILY_SELLER_SUMMARY table

The task will write into this table. Create it before the stream and task so the task has a target to INSERT into.


In [ ]:
-- Target table for the nightly aggregation task
-- One row per seller per day — accumulated over time as the task runs
CREATE OR REPLACE TABLE SHOPFLOW_ANALYTICS.DAILY_SELLER_SUMMARY (
    seller_id       VARCHAR,
    summary_date    DATE,
    orders_count    INTEGER,
    items_count     INTEGER,
    gmv_brl         FLOAT,
    avg_price_brl   FLOAT,
    loaded_at       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Nightly seller performance summary — populated by NIGHTLY_SELLER_SUMMARY task';

SELECT 'DAILY_SELLER_SUMMARY created' AS status;


---
## Cell 3 — Create the stream on FACT_ORDERS


In [ ]:
-- Create an append-only stream on FACT_ORDERS
-- APPEND_ONLY = TRUE is correct here — orders are never updated or deleted
-- More efficient than a standard stream for insert-only tables
CREATE OR REPLACE STREAM SHOPFLOW_ANALYTICS.ORDERS_STREAM
    ON TABLE SHOPFLOW_ANALYTICS.FACT_ORDERS
    APPEND_ONLY = TRUE
    COMMENT     = 'CDC stream on FACT_ORDERS — tracks new order items for nightly aggregation';

-- Confirm stream was created
SHOW STREAMS IN SCHEMA SHOPFLOW_ANALYTICS;


---
## Cell 4 — Inspect the stream

Query the stream to see what it contains. Right after creation, the stream is empty — no changes have happened since it was created. This is correct and expected.

The three `METADATA$` columns are visible here. These are automatically added to every stream query — you don't define them.


In [ ]:
-- Preview stream contents — shows rows changed since stream creation
-- Should be empty right now (no new rows inserted into FACT_ORDERS yet)
SELECT
    METADATA$ACTION,
    METADATA$ISUPDATE,
    order_id,
    seller_id,
    price,
    order_purchase_ts
FROM SHOPFLOW_ANALYTICS.ORDERS_STREAM
LIMIT 10;


In [ ]:
-- Check if stream has any unconsumed records
-- Returns FALSE right now — no data has been inserted since stream creation
-- This is the same function used in the task WHEN clause
SELECT SYSTEM$STREAM_HAS_DATA(
    'SHOPFLOW_DB.SHOPFLOW_ANALYTICS.ORDERS_STREAM'
) AS stream_has_data;


---
## Cell 5 — Simulate new orders arriving

Insert a test row into `FACT_ORDERS` to give the stream something to track. After this, `SYSTEM$STREAM_HAS_DATA` returns `TRUE` and the stream shows the new row.

> **💡 Important — METADATA$ACTION behaviour:**
>
> When you INSERT a row, the stream records it with `METADATA$ACTION = 'INSERT'`. If you then UPDATE that row, the stream records two rows: one `DELETE` (the old value) and one `INSERT` (the new value). This is how Snowflake represents updates — as a delete + insert pair. For append-only streams, only `INSERT` is possible.


In [ ]:
-- Insert a test row to give the stream something to track
-- Use a clearly fake order_id so it is easy to identify and clean up
INSERT INTO SHOPFLOW_ANALYTICS.FACT_ORDERS (
    order_id, order_item_id, product_id, seller_id,
    customer_id, order_status, order_purchase_ts,
    price, freight_value, total_item_value,
    delivery_delay_days, payment_value,
    primary_payment_type, max_installments
)
VALUES (
    'TEST_ORDER_STREAM_001', 1, 'TEST_PRODUCT', 'TEST_SELLER',
    'TEST_CUSTOMER', 'delivered', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
    99.90, 12.50, 112.40,
    -2, 112.40,
    'credit_card', 1
);

SELECT 'Test row inserted into FACT_ORDERS' AS status;


In [ ]:
-- Now the stream should show the new row
SELECT SYSTEM$STREAM_HAS_DATA(
    'SHOPFLOW_DB.SHOPFLOW_ANALYTICS.ORDERS_STREAM'
) AS stream_has_data;


In [ ]:
-- Preview the stream — should now show the test row with METADATA$ACTION = 'INSERT'
SELECT
    METADATA$ACTION,
    METADATA$ISUPDATE,
    order_id,
    seller_id,
    price,
    order_purchase_ts
FROM SHOPFLOW_ANALYTICS.ORDERS_STREAM;


---
## Cells 6–9 — Tasks (Scheduled SQL Automation)

### What is a task?

A **task** is a scheduled SQL statement — or a call to a stored procedure — that runs automatically on a CRON schedule or when triggered by another task. No Lambda, no Airflow, no external scheduler needed.

```sql
CREATE TASK my_task
    WAREHOUSE = my_wh          -- which warehouse runs this
    SCHEDULE  = 'USING CRON ...'  -- when to run
    WHEN <condition>           -- optional: only run if condition is TRUE
AS
    <SQL statement>;           -- what to execute
```

### Warehouse tasks vs serverless tasks

| | Warehouse task | Serverless task |
|--|----------------|-----------------|
| Compute | Your named warehouse | Snowflake manages it |
| How to specify | `WAREHOUSE = SHOPFLOW_WH` | `USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'` |
| Billing | Credits from your warehouse | Compute credits billed separately |
| Good for | Predictable, regular workloads | Irregular or lightweight workloads |

We use a warehouse task — simpler for a learning project.

### CRON syntax in Snowflake

```text
SCHEDULE = 'USING CRON <minute> <hour> <day> <month> <weekday> <timezone>'

Examples:
  'USING CRON 0 1 * * * Asia/Kolkata'    → 1:00 AM every day IST
  'USING CRON 0 */6 * * * UTC'           → every 6 hours UTC
  'USING CRON 0 9 * * MON UTC'           → 9:00 AM every Monday UTC
  'USING CRON 30 8 1 * * America/New_York' → 8:30 AM on the 1st of every month ET
```

### Task DAGs — chaining tasks

Tasks can be chained: Task B runs after Task A completes successfully. This creates a Directed Acyclic Graph (DAG) — the same concept as Airflow DAGs but defined entirely in SQL:

```sql
CREATE TASK task_b
    AFTER task_a          -- triggers when task_a succeeds
AS
    <SQL>;
```

No scheduler, no orchestration tool, no container. Pure SQL.

> **🎯 SnowPro Tip: Task Rules**
>
> Three rules that are heavily tested:
> 1. Tasks are **SUSPENDED by default** after `CREATE TASK` — you must `ALTER TASK ... RESUME` explicitly
> 2. A task with a `WHEN` clause only executes the SQL if the condition returns `TRUE` — if `FALSE`, the task is skipped but counts as a successful run
> 3. `TASK_HISTORY` in `INFORMATION_SCHEMA` shows the last 7 days of task run history — state, error message, duration


---
## Cell 6 — Create the nightly aggregation task

This task:
1. Checks if `ORDERS_STREAM` has new data (`WHEN` clause)
2. If yes — reads from the stream and aggregates into `DAILY_SELLER_SUMMARY`
3. Reading from the stream inside the INSERT advances the stream offset
4. Runs at 01:00 AM IST every night

> **💡 Why the stream must be read inside a DML statement:**
>
> The INSERT...SELECT is the DML that consumes the stream. If you ran a plain SELECT inside the task, the stream offset would not advance — the same rows would appear again on the next run. The INSERT is what commits the consumption.


In [ ]:
-- Create the nightly aggregation task
-- Reads from ORDERS_STREAM and writes to DAILY_SELLER_SUMMARY
CREATE OR REPLACE TASK SHOPFLOW_ANALYTICS.NIGHTLY_SELLER_SUMMARY
    WAREHOUSE = SHOPFLOW_WH
    SCHEDULE  = 'USING CRON 0 1 * * * Asia/Kolkata'
    WHEN SYSTEM$STREAM_HAS_DATA(
        'SHOPFLOW_DB.SHOPFLOW_ANALYTICS.ORDERS_STREAM'
    )
AS
    INSERT INTO SHOPFLOW_ANALYTICS.DAILY_SELLER_SUMMARY (
        seller_id,
        summary_date,
        orders_count,
        items_count,
        gmv_brl,
        avg_price_brl
    )
    SELECT
        seller_id,
        DATE_TRUNC('day', order_purchase_ts)::DATE  AS summary_date,
        COUNT(DISTINCT order_id)                    AS orders_count,
        COUNT(*)                                    AS items_count,
        ROUND(SUM(price + freight_value), 2)        AS gmv_brl,
        ROUND(AVG(price), 2)                        AS avg_price_brl
    FROM SHOPFLOW_ANALYTICS.ORDERS_STREAM
    WHERE METADATA$ACTION = 'INSERT'
    GROUP BY seller_id, summary_date;

-- Confirm task was created — note state = 'Suspended'
SHOW TASKS IN SCHEMA SHOPFLOW_ANALYTICS;


---
## Cell 7 — Resume the task

Every new task starts in `SUSPENDED` state. Run `ALTER TASK ... RESUME` to activate it. Until you resume it, the CRON schedule does nothing.


In [ ]:
-- Activate the task — it will now run on its CRON schedule
ALTER TASK SHOPFLOW_ANALYTICS.NIGHTLY_SELLER_SUMMARY RESUME;

-- Confirm state changed to 'Started'
SHOW TASKS IN SCHEMA SHOPFLOW_ANALYTICS;


---
## Cell 8 — Execute the task manually

We don't want to wait until 1:00 AM IST to see the task run. `EXECUTE TASK` triggers it immediately — same logic as the scheduled run, just on demand.

After execution, check `DAILY_SELLER_SUMMARY` for the aggregated row from our test order, and check that the stream is now empty (offset advanced).


In [ ]:
-- Trigger the task immediately — don't wait for the CRON schedule
EXECUTE TASK SHOPFLOW_ANALYTICS.NIGHTLY_SELLER_SUMMARY;

SELECT 'Task executed — check DAILY_SELLER_SUMMARY and stream below' AS status;


In [ ]:
-- Check DAILY_SELLER_SUMMARY — should contain the TEST_SELLER row
SELECT *
FROM SHOPFLOW_ANALYTICS.DAILY_SELLER_SUMMARY
ORDER BY loaded_at DESC;


In [ ]:
-- Stream should now be empty — the INSERT consumed the offset
SELECT SYSTEM$STREAM_HAS_DATA(
    'SHOPFLOW_DB.SHOPFLOW_ANALYTICS.ORDERS_STREAM'
) AS stream_has_data;

-- Also confirm stream shows no rows
SELECT COUNT(*) AS unconsumed_rows
FROM SHOPFLOW_ANALYTICS.ORDERS_STREAM;


---
## Cell 9 — Check task run history

`INFORMATION_SCHEMA.TASK_HISTORY` shows every task execution in the last 7 days — state, duration, error message if failed, and whether the `WHEN` condition was met.


In [ ]:
-- Task run history — shows state, duration, and any errors
SELECT
    NAME                                        AS task_name,
    STATE,
    TO_CHAR(SCHEDULED_TIME, 'YYYY-MM-DD HH24:MI:SS') AS scheduled_at,
    TO_CHAR(QUERY_START_TIME,'HH24:MI:SS')      AS started_at,
    COMPLETED_TIME - QUERY_START_TIME           AS duration,
    ERROR_CODE,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP()),
    TASK_NAME => 'NIGHTLY_SELLER_SUMMARY'
))
ORDER BY SCHEDULED_TIME DESC;


---
## Cell 10 — Suspend the task

Suspend the task after testing so it doesn't run at 1:00 AM tonight and charge credits unnecessarily. On a trial account every credit matters.

> **💡 Cost tip:** A task that runs on a CRON schedule starts the warehouse for each execution. Even a 1-second task uses the 60-second minimum billing increment. On `SHOPFLOW_WH` (X-Small = 1 credit/hour), a task that runs nightly costs ~0.017 credits per execution. Suspend when not needed.


In [ ]:
-- Suspend the task — stops it from running on the CRON schedule
ALTER TASK SHOPFLOW_ANALYTICS.NIGHTLY_SELLER_SUMMARY SUSPEND;

-- Confirm state = 'Suspended'
SHOW TASKS IN SCHEMA SHOPFLOW_ANALYTICS;


---
## Cells 11–13 — Secure Data Sharing

### What is a secure share?

A **secure share** is a Snowflake object that exposes selected tables or views to another Snowflake account. The consumer account queries your data live — their queries run against **your storage**. No data is ever copied. No pipeline. No S3. No file exports.

```text
Provider account (you)          Consumer account (partner)
┌──────────────────────┐        ┌──────────────────────┐
│  DAILY_SELLER_SUMMARY│        │  Queries your table  │
│         ↑            │◄──────►│  as if it were local │
│  BRAND_PARTNER_SHARE │        │  0 data movement     │
└──────────────────────┘        └──────────────────────┘
         your storage                  their compute
```

### What makes it "secure"

- The consumer cannot see your other tables — only what you explicitly grant to the share
- The consumer cannot modify data — read-only access only
- You can revoke the share at any time — access is immediately removed
- No API keys, no credentials, no VPN — Snowflake handles the authentication between accounts

### Snowflake Marketplace

The Marketplace is the public version of data sharing — companies publish datasets (weather, financial data, demographic data) as shares that any Snowflake account can subscribe to. The data stays on the provider's storage. Consumers pay for their own compute to query it.

> **🎯 SnowPro Tip: Sharing Rules**
>
> Key facts for the exam:
> - Sharing works only between **Snowflake accounts** — the consumer must have a Snowflake account
> - You can share **tables and views** but not stages, file formats, or tasks
> - The provider account is billed for **storage** of shared data. The consumer account is billed for **compute** used to query it
> - `CREATE SHARE` and `SHOW SHARES` require `ACCOUNTADMIN` or `SYSADMIN` role
> - A **reader account** is a free lightweight account Snowflake creates for consumers who don't have their own account


---
## Cell 11 — Create the share

Three grant steps are always required in sequence — each one unlocks the next level:

1. `GRANT USAGE ON DATABASE` — consumer can see the database
2. `GRANT USAGE ON SCHEMA` — consumer can see the schema
3. `GRANT SELECT ON TABLE` — consumer can query the specific table

Missing any one of these blocks access even if the others are granted.


In [ ]:
USE ROLE SYSADMIN;

-- Create the share object
CREATE SHARE IF NOT EXISTS SHOPFLOW_ANALYTICS.BRAND_PARTNER_SHARE
    COMMENT = 'Live seller summary data for brand partners — no data copy';

-- Step 1: database access
GRANT USAGE ON DATABASE SHOPFLOW_DB
    TO SHARE BRAND_PARTNER_SHARE;

-- Step 2: schema access
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS
    TO SHARE BRAND_PARTNER_SHARE;

-- Step 3: table access — only DAILY_SELLER_SUMMARY, not FACT_ORDERS
GRANT SELECT ON TABLE SHOPFLOW_DB.SHOPFLOW_ANALYTICS.DAILY_SELLER_SUMMARY
    TO SHARE BRAND_PARTNER_SHARE;

SELECT 'Share created with 3 grant levels' AS status;


---
## Cell 12 — Verify the share


In [ ]:
-- List all shares on this account
-- You should see BRAND_PARTNER_SHARE with kind = 'OUTBOUND'
SHOW SHARES;


In [ ]:
-- See exactly what is granted to the share
-- Should show: USAGE on DATABASE, USAGE on SCHEMA, SELECT on TABLE
SHOW GRANTS TO SHARE BRAND_PARTNER_SHARE;


---
## Cell 13 — Add a consumer account (informational)

To actually give a partner access, you add their Snowflake account locator to the share. On a trial account you likely don't have a second account to test with — but the syntax is important to know for the exam.

```sql
-- Add a consumer account — they can now query the share
ALTER SHARE BRAND_PARTNER_SHARE
    ADD ACCOUNTS = <partner_account_locator>;

-- Remove access
ALTER SHARE BRAND_PARTNER_SHARE
    REMOVE ACCOUNTS = <partner_account_locator>;
```

On the **consumer side**, they create a database from the share:

```sql
-- Consumer runs this on their account
CREATE DATABASE seller_portal_db
    FROM SHARE <your_account>.<share_name>;

-- Then queries it like any local table
SELECT * FROM seller_portal_db.shopflow_analytics.daily_seller_summary;
```

The consumer's query runs against your storage. Your data never moved. Their query uses their compute credits — not yours.


---
## Cell 14 — Clean up test data

Remove the test row inserted in Cell 5. It served its purpose — demonstrated stream tracking — but should not live in `FACT_ORDERS` permanently.

> **💡 Note:** Deleting from `FACT_ORDERS` will create a new stream entry (`METADATA$ACTION = 'DELETE'`). Since the stream is append-only, this delete will not appear in the stream — append-only streams only track inserts. The stream stays clean.


In [ ]:
-- Remove the test order inserted in Cell 5
DELETE FROM SHOPFLOW_ANALYTICS.FACT_ORDERS
WHERE order_id = 'TEST_ORDER_STREAM_001';

-- Remove the test summary row
DELETE FROM SHOPFLOW_ANALYTICS.DAILY_SELLER_SUMMARY
WHERE seller_id = 'TEST_SELLER';

-- Confirm FACT_ORDERS row count is back to the pre-test count
SELECT COUNT(*) AS total_fact_rows FROM SHOPFLOW_ANALYTICS.FACT_ORDERS;


---
## Cell 15 — Suspend warehouse


In [ ]:
ALTER WAREHOUSE SHOPFLOW_WH SUSPEND;


In [ ]:
SHOW WAREHOUSES LIKE 'SHOPFLOW%';


---
## 🛠 Self-Guided Exploration:

Now that the stream, task, and share are all working — test your understanding. Write these from scratch.

### Challenge 1 (Easy): Inspect the stream object
**Objective:** Practice reading stream metadata.

**Task:** Write the `SHOW` command that lists all streams in `SHOPFLOW_ANALYTICS`. From the output, identify: what is the `stale` column telling you, and what happens to a stream that becomes stale?

> 💡 **Think about it:** A stream becomes stale when its `STALE_AFTER` timestamp has passed — typically 14 days after the last time data was consumed. What happens to the stream offset when it goes stale?


In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2 (Medium): Query task history for all tasks
**Objective:** Master TASK_HISTORY for operational monitoring.

**Task:** Write a query against `INFORMATION_SCHEMA.TASK_HISTORY` that shows all task executions in the last 24 hours across all tasks in the schema — not just `NIGHTLY_SELLER_SUMMARY`. Show task name, state, scheduled time, and duration. Order by most recent first.

> 💡 **Think about it:** What `STATE` values are possible in TASK_HISTORY? What is the difference between `SUCCEEDED` and `SKIPPED`?


In [ ]:
-- Write your SQL for Challenge 2 here


### Challenge 3 (Hard): Create a chained task DAG
**Objective:** Build a two-task pipeline where Task B runs after Task A completes.

**Task:** Create a second task `CLEANUP_OLD_SUMMARIES` that runs **after** `NIGHTLY_SELLER_SUMMARY` completes. This task should delete rows from `DAILY_SELLER_SUMMARY` where `summary_date` is older than 90 days. Chain it using `AFTER`. Resume both tasks and verify the dependency in `SHOW TASKS`.

> 💡 **Hint:** A chained task uses `AFTER task_name` instead of `SCHEDULE`. It has no schedule of its own — it runs when its predecessor succeeds. Remember to `SUSPEND` both tasks when done.

> ⚠️ **Watch out:** The root task (the one with the SCHEDULE) must be suspended before you can modify any task in its DAG. Snowflake errors if you try to `ALTER` a task whose root task is running.


In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4 (Complex): Reconstruct what the task inserted
**Objective:** Understand stream consumption by replaying the logic manually.

**Task:** Without running the task again, write the same SELECT that the task uses internally — but instead of reading from `ORDERS_STREAM`, read from `FACT_ORDERS` directly and filter for today's inserted rows using `order_purchase_ts`. Aggregate by `seller_id` and `summary_date`. Compare the result to what's currently in `DAILY_SELLER_SUMMARY`.

> 💡 **Think about it:** The stream and a direct table query should produce the same result for today's data. If they don't, what could cause the difference? Think about what data the stream saw vs what is currently in the table.


In [ ]:
-- Write your SQL for Challenge 4 here


---
## Phase 5 complete

If Cell 5 showed the stream tracking the test row, Cell 8 showed `DAILY_SELLER_SUMMARY` populated after `EXECUTE TASK`, Cell 9 showed a `SUCCEEDED` task run, and Cell 12 showed the share with 3 grants — your automation and sharing pipeline is fully operational.

### What you just built

```text
SHOPFLOW_ANALYTICS
├── ORDERS_STREAM           ← append-only CDC on FACT_ORDERS
├── DAILY_SELLER_SUMMARY    ← aggregated nightly output table
├── NIGHTLY_SELLER_SUMMARY  ← task · CRON 01:00 IST · WHEN stream has data
└── BRAND_PARTNER_SHARE     ← live share · DAILY_SELLER_SUMMARY only
```

### The full pipeline in 4 SQL statements

```sql
CREATE STREAM  ORDERS_STREAM ON TABLE FACT_ORDERS APPEND_ONLY = TRUE;
CREATE TABLE   DAILY_SELLER_SUMMARY (...);
CREATE TASK    NIGHTLY_SELLER_SUMMARY ... AS INSERT INTO ... SELECT ... FROM ORDERS_STREAM;
CREATE SHARE   BRAND_PARTNER_SHARE;
```

What this replaces in AWS: DynamoDB Streams + Kinesis + Lambda + S3 + Glue + Redshift + API Gateway.

### Key Snowflake concepts you practised

| Topic | Covered |
|-------|---------|
| CREATE STREAM — append-only vs standard | ✅ |
| METADATA$ACTION, METADATA$ISUPDATE | ✅ |
| Stream offset — advances on DML only | ✅ |
| SYSTEM$STREAM_HAS_DATA() | ✅ |
| CREATE TASK — warehouse tasks | ✅ |
| CRON schedule syntax + timezones | ✅ |
| WHEN clause — conditional execution | ✅ |
| ALTER TASK RESUME / SUSPEND | ✅ |
| EXECUTE TASK — manual trigger | ✅ |
| INFORMATION_SCHEMA.TASK_HISTORY | ✅ |
| Task DAGs — AFTER clause | ✅ |
| CREATE SHARE — 3-level grant hierarchy | ✅ |
| SHOW GRANTS TO SHARE | ✅ |
| Provider vs consumer billing model | ✅ |

### ShopFlow project — complete ✅

All 6 phases done. Here is what you built end to end:

```text
Kaggle CSVs
    ↓  Phase 00 — Snowflake setup
@OLIST_STAGE
    ↓  Phase 01 — COPY INTO
SHOPFLOW_RAW (9 tables, all VARCHAR)
    ↓  Phase 02 — TRY_CAST, QUALIFY, COALESCE
SHOPFLOW_STAGED (9 tables, typed + clean)
    ↓  Phase 03 — Star schema
SHOPFLOW_ANALYTICS
    ├── FACT_ORDERS + 4 DIM tables
    ├── 4 answered business queries
    └── Clustering key on FACT_ORDERS
    ↓  Phase 04 — Security
    ├── 3 RBAC roles
    ├── Dynamic data masking
    └── Row access policy
    ↓  Phase 05 — Automation
    ├── ORDERS_STREAM (CDC)
    ├── NIGHTLY_SELLER_SUMMARY task
    ├── DAILY_SELLER_SUMMARY table
    └── BRAND_PARTNER_SHARE
```

### Next steps

- Push all notebooks to GitHub → `https://github.com/bh00t/ShopFlow-Analytics`
- Write a polished `README.md` — what you built, why, what you learned
- Schedule and sit the **SnowPro Core exam** — you have covered ~85% of the exam domains hands-on
- Update the ShopFlow website — mark Phases 03–05 as complete
